Import reddit data as a .jsonl file, transform into a csv/data frame for ease of use

In [ ]:
# load json file as a csv
import pandas as pd
import json
import os

filename = '/content/r_Drexel_posts.jsonl'  #json lines file, subreddit creation - 07/01/2026
output_folder = 'content/output'

with open(filename) as f:
    lines = f.read().splitlines()

my_dict = {}
for i, line in enumerate(lines):
    try:
        my_dict[i] = json.loads(line)
    except:
        pass

df_raw = pd.DataFrame.from_dict(my_dict).T
df_raw.shape


(32053, 134)

In [ ]:
df_raw.head()

,archived,author,author_flair_background_color,author_flair_css_class,author_flair_richtext,author_flair_text,author_flair_text_color,author_flair_type,brand_safe,can_gild,...,is_gallery,call_to_action,author_is_blocked,_meta,previous_selftext,location_lat,location_long,location_name,websocket_url,outbound_link
0,True,brtw,None,textflair,"[{'e': 'text', 't': 'Alumni | Cheese'}]",Alumni | Cheese,None,richtext,True,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,True,brtw,None,textflair,"[{'e': 'text', 't': 'Alumni | Cheese'}]",Alumni | Cheese,None,richtext,True,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,True,[deleted],,None,NaN,None,dark,NaN,True,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,True,Sporknight,None,None,[],None,None,text,True,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,True,kjhobin,None,textflair,"[{'e': 'text', 't': 'Information Systems Alum ...",Information Systems Alum '13,None,richtext,True,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# remove deleted, removed posts
df_raw = df_raw[~df_raw["author"].isin(["[removed]", "[deleted]", ""])]
df_raw = df_raw[~df_raw["selftext"].isin(["[removed]", "[deleted]", ""])]

# keep only relavent columns
df_filtered = df_raw[["id", "author", "created_utc", "title", "selftext"]].copy()

#combine title and selftext into one field
df_filtered["text"] = df_filtered["title"].fillna("") + " " + df_filtered["selftext"].fillna("")

df_filtered.shape

(22132, 6)

In [ ]:
# clean up the reddit data
import re

def clean_reddit_text(text):
    text = re.sub(r'http\S+|www\.\S+', ' ', text)          # URLs
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)   # [text](link) -> text
    text = re.sub(r'&amp;|&gt;|&lt;', ' ', text)             # HTML entities
    text = re.sub(r'\*{1,2}([^\*]+)\*{1,2}', r'\1', text)    # **bold**/*italic*
    text = re.sub(r'/r/\w+|/u/\w+|r/\w+|u/\w+', ' ', text)   # subreddit/user mentions
    text = re.sub(r'\n+', ' ', text)                         # newlines
    return text

df_filtered["text_clean"] = df_filtered["text"].apply(clean_reddit_text)

In [ ]:
# tokenzie and lemmatize
import spacy
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

docs = df_filtered["text_clean"].tolist()

tokens_list = []
for doc in nlp.pipe(docs, batch_size=100):
    tokens_list.append([
        token.lemma_.lower() for token in doc
        if token.is_alpha and not token.is_stop and len(token) > 2
    ])

df_filtered["tokens"] = tokens_list

In [ ]:
from collections import Counter

# combined stopword list first and second wave of words
custom_stopwords = {
    "drexel", "know", "like", "look", "want", "get", "thank",
    "take", "need", "time", "go", "good", "help", "find",
    "think", "question", "wonder", "try", "guy", "hey", "pay"
}

custom_stopwords.update({
    "people", "term", "experience", "plan", "live", "interested",
    "offer", "start", "week", "come", "say", "feel", "hear", "place",
    "ask", "new", "things", "currently", "way", "day", "thing", "lot", "let"
})

# apply filter to token columns
df_filtered["tokens"] = df_filtered["tokens"].apply(
    lambda tokens: [t for t in tokens if t not in custom_stopwords]
)

# 3. rebuild tokens from the filtered column and recheck frequencies
all_tokens = [t for tokens in df_filtered["tokens"] for t in tokens]
Counter(all_tokens).most_common(40)

[('class', 6474),
 ('student', 6070),
 ('year', 4876),
 ('work', 3404),
 ('major', 3204),
 ('campus', 3174),
 ('program', 2930),
 ('course', 2555),
 ('school', 2517),
 ('fall', 2210),
 ('room', 1986),
 ('freshman', 1889),
 ('job', 1827),
 ('university', 1800),
 ('email', 1689),
 ('apply', 1669),
 ('study', 1637),
 ('summer', 1636),
 ('online', 1628),
 ('coop', 1628),
 ('engineering', 1626),
 ('housing', 1609),
 ('college', 1557),
 ('professor', 1539),
 ('month', 1418),
 ('quarter', 1413),
 ('interview', 1388),
 ('spring', 1384),
 ('apartment', 1381),
 ('advice', 1342),
 ('roommate', 1342),
 ('accept', 1310),
 ('free', 1308),
 ('credit', 1302),
 ('transfer', 1269),
 ('lease', 1254),
 ('graduate', 1246),
 ('aid', 1210),
 ('tell', 1196),
 ('friend', 1194)]

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 47.4 MB/s eta 0:00:00


In [ ]:
# build the dictionary and corpus
from gensim import corpora

dictionary = corpora.Dictionary(df_filtered["tokens"])
dictionary.filter_extremes(no_below=5, no_above=0.5)
corpus = [dictionary.doc2bow(tokens) for tokens in df_filtered["tokens"]]

In [ ]:
# decide which k to use
from gensim.models import LdaModel, CoherenceModel

for k in [2, 3, 4, 5, 6, 7]:
    lda = LdaModel(corpus=corpus, id2word=dictionary, num_topics=k, passes=10, random_state=42)
    cm = CoherenceModel(model=lda, texts=df_filtered["tokens"], dictionary=dictionary, coherence='c_v')
    print(k, cm.get_coherence())

2 0.5520121768239954
3 0.5292881980302203
4 0.5070250358766453
5 0.5472936043059631
6 0.5446313987406487
7 0.551150231655695


In [ ]:
# use K=7
lda_7 = LdaModel(corpus=corpus, id2word=dictionary, num_topics=7, passes=15, random_state=42)
for idx, topic in lda_7.print_topics(-1):
    print(f"Topic {idx}: {topic}")

Topic 0: 0.015*"comcast" + 0.012*"campus" + 0.012*"student" + 0.010*"friend" + 0.008*"club" + 0.007*"food" + 0.007*"group" + 0.006*"open" + 0.006*"work" + 0.006*"hour"
Topic 1: 0.030*"interview" + 0.030*"job" + 0.027*"coop" + 0.027*"work" + 0.027*"major" + 0.022*"round" + 0.022*"engineering" + 0.018*"accept" + 0.017*"engineer" + 0.016*"employer"
Topic 2: 0.039*"student" + 0.034*"program" + 0.033*"year" + 0.026*"school" + 0.015*"college" + 0.014*"aid" + 0.014*"transfer" + 0.014*"international" + 0.012*"apply" + 0.012*"financial"
Topic 3: 0.027*"room" + 0.025*"apartment" + 0.024*"lease" + 0.022*"roommate" + 0.021*"campus" + 0.019*"housing" + 0.017*"month" + 0.015*"rent" + 0.015*"summit" + 0.012*"bed"
Topic 4: 0.068*"class" + 0.030*"course" + 0.016*"professor" + 0.016*"exam" + 0.015*"online" + 0.013*"quarter" + 0.012*"credit" + 0.012*"math" + 0.010*"grade" + 0.010*"summer"
Topic 5: 0.019*"sell" + 0.015*"ticket" + 0.015*"sale" + 0.013*"buy" + 0.010*"free" + 0.010*"survey" + 0.009*"item" + 